# Air Quality Sensor Drift Intelligence
### Exploratory Data Analysis, Multi-Sensor Drift Surveillance & Degradation Modeling

**Dataset**: UCI Gas Sensor Array Drift Dataset  
**Array Architecture**: 16 Figaro Chemical Sensors, 128 Extracted Features, 6 Gas Classes, 10 Chronological Batches (~36 Months).

This notebook provides transparent, reproducible data science workflows for exploring the dataset, validating chronological scaling, observing PCA latent drift, demonstrating concept drift degradation on classifiers, and computing statistical drift (PSI & KS tests).

In [ ]:
import sys
from pathlib import Path
project_root = Path.cwd().parent
if str(project_root) not in sys.path:
    sys.path.append(str(project_root))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.decomposition import PCA
from sklearn.ensemble import RandomForestClassifier, IsolationForest
from sklearn.metrics import accuracy_score, classification_report

from src.data_loader import load_processed_data
from src.preprocessing import DataQualityChecker, ChronologicalPreprocessor
from src.drift_detection import calculate_psi, calculate_ks_test, compute_all_features_drift, compute_sensor_level_drift_summary
from src.sensor_health import SensorHealthScorer

print("✓ All project modules imported successfully!")

## 1. Data Ingestion & Quality Diagnostics
We load all 13,910 measurements across 10 chronological batches and verify quality.

In [ ]:
df = load_processed_data()
print(f"Consolidated Dataset: {df.shape[0]} rows, {df.shape[1]} columns.")
report = DataQualityChecker.inspect_dataframe(df)
print("Quality Report:", report)
df.head()

## 2. Gas Analyte & Batch Distributions

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

sns.countplot(data=df, x='gas_name', ax=axes[0], palette='viridis')
axes[0].set_title('Gas Class Sample Distribution')
axes[0].set_xlabel('Chemical Analyte')
axes[0].set_ylabel('Sample Count')
axes[0].tick_params(axis='x', rotation=25)

sns.countplot(data=df, x='batch_id', ax=axes[1], palette='Blues_r')
axes[1].set_title('Measurement Frequency per Batch (~36 Months)')
axes[1].set_xlabel('Chronological Batch ID')
axes[1].set_ylabel('Record Count')

plt.tight_layout()
plt.show()

## 3. Chronological Scaling (Leakage-Free)
The preprocessor fits strictly on Batch 1 (the initial calibration epoch) and transforms subsequent operational batches without lookahead bias.

In [ ]:
prep = ChronologicalPreprocessor(scaler_type='standard')
train_df, test_df = prep.fit_transform_chronological(df, train_batch_ids=[1])
scaled_df = prep.transform(df)
print(f"Train Batch 1 samples: {len(train_df)}, Test Batches 2-10 samples: {len(test_df)}")

## 4. PCA Latent Manifold Drift
PCA fitted on Batch 1 captures the initial calibration manifold. Projecting later batches reveals continuous spatial drift over time.

In [ ]:
feat_cols = [f'f{i}' for i in range(1, 129)]
pca = PCA(n_components=2, random_state=42)
pca.fit(train_df[feat_cols])

coords = pca.transform(scaled_df[feat_cols])
pca_df = pd.DataFrame(coords, columns=['PC1', 'PC2'])
pca_df['batch_id'] = scaled_df['batch_id']
pca_df['gas_name'] = scaled_df['gas_name']

plt.figure(figsize=(10, 6))
sns.scatterplot(data=pca_df, x='PC1', y='PC2', hue='batch_id', palette='coolwarm', alpha=0.6)
plt.title('2D PCA Projection: Visualizing Multi-Sensor Drift across Batches')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]*100:.1f}% var)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]*100:.1f}% var)')
plt.show()

## 5. Machine Learning & Progressive Degradation
Demonstrating concept drift: A Random Forest trained on Batch 1 degrades in gas classification accuracy over subsequent chronological batches.

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(train_df[feat_cols], train_df['gas_class'])

deg_records = []
for b in range(1, 11):
    b_data = scaled_df[scaled_df['batch_id'] == b]
    preds = rf.predict(b_data[feat_cols])
    acc = accuracy_score(b_data['gas_class'], preds)
    deg_records.append({'batch_id': b, 'accuracy': acc, 'samples': len(b_data)})

deg_df = pd.DataFrame(deg_records)
print(deg_df)

plt.figure(figsize=(9, 4))
plt.plot(deg_df['batch_id'], deg_df['accuracy'] * 100, marker='o', color='#1E88E5', linewidth=2.5)
plt.title('Progressive Accuracy Degradation Over 36 Months')
plt.xlabel('Chronological Batch ID')
plt.ylabel('Accuracy (%)')
plt.grid(True, linestyle='--', alpha=0.5)
plt.show()

## 6. Statistical Drift Detection & Sensor Reliability Scoring
We evaluate feature drift using Population Stability Index (PSI) and Kolmogorov-Smirnov test between Batch 1 and Batch 10, then generate explainable sensor health scores.

In [ ]:
ref_b = df[df['batch_id'] == 1]
tgt_b = df[df['batch_id'] == 10]

feat_drifts = compute_all_features_drift(ref_b, tgt_b)
sensor_drift = compute_sensor_level_drift_summary(feat_drifts)
health_df = SensorHealthScorer.calculate_health_scores(sensor_drift)

print("16-Sensor Health & Reliability Summary (Batch 1 vs Batch 10):")
health_df[['Sensor_ID', 'Sensor_Type', 'Health_Score', 'Category', 'Recommendation']]